# 03_diagnosis_recheck_CHS

- 목적: 02 진단의 결론 중 뒤집힐 수 있는 세 가지를 재검증한다 — A-1 DC 오프셋의 정체(에일리어싱 산물인가), A-2 진동 스케일 동일성, A-3 형식 누수 차단 증명
- 입력: `data/raw/` 원본 2파일(`data_quality.load`), `data/processed/segments.csv`(`segments.build_segments_table`)
- 출력: `figures/03_diagnosis_recheck_CHS/`, `results/03_diagnosis_recheck_CHS/`
- 담당: CHS
- 공통 모듈: `src/preprocess.py`(형식 통일·DC 제거·형식 피처), `src/split.py`(세그먼트 GroupKFold·시간 블록), `src/signal_checks.py`(사인 피팅)


In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
NB = "03_diagnosis_recheck_CHS"  # 파일명(확장자 제외)과 같게
FIG, RES = paths.nb_dirs(NB)
SEED = 42
np.random.seed(SEED)

In [2]:
import data_quality as dq, preprocess as pp, split as sp, signal_checks as sc, segments as sg
from scipy import stats
from sklearn.metrics import roc_auc_score
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

dfs = dq.load_all()
raw = pp.add_seg_uid(pd.concat([dfs["normal"], dfs["outlier"]], ignore_index=True))
seg = sg.build_segments_table(save=True)            # data/processed/segments.csv 재생성 (620 × 26)
seg_meta = seg.set_index("seg_uid")[["src", "seg_id", "len", "label", "state", "vib_grade", "cur_grade", "vib_rms"]]
print(raw.shape, seg.shape, raw["seg_uid"].nunique())

(20600, 10) (620, 26) 620


## A-1. DC 오프셋의 정체 — 에일리어싱 산물인가, 실제 오프셋인가

**문제.** 02 §2에서 세그먼트 평균(DC 오프셋) 하나로 정상/이상을 AUC 0.89~0.94로 구분할 수 있어 "파일 형식 누수"로 결론했고, "DC 제거 필수"를 전처리 규칙으로 잡았다. 그런데 전류는 60 Hz 구동이 10 Hz 샘플링에서 접힌 신호다. 구동 주파수가 60 Hz의 정수배 근처에서 조금 벗어나면 접힌 성분은 0.1 Hz 이하의 느린 정현파가 되고, 5초 burst 안에서는 **DC처럼** 보인다. 그렇다면 DC는 형식 누수가 아니라 "구동 주파수 이탈"이라는 물리 신호일 수 있다.

**검증.** 세그먼트마다 `A·sin(2πft+φ)+C`를 최소제곱으로 맞춘다(`signal_checks.sine_fit_segment`, f는 0.01~5 Hz 로그 격자 500점 + 정밀화).
- 가설이 맞으면: 피팅 f가 0.1 Hz 이하로 갈수록 |DC|가 커진다 → 전류 |DC| vs f의 Spearman |rho| ≥ 0.5.
- 이상 세그먼트 16~20(DC 180~272)의 피팅 진폭 A가 정상 수준(약 100~270)이고 f≈0.02~0.1 Hz면 "구동 주파수 이탈", A가 정상보다 작고 C만 크면 "실제 오프셋".

**판정 규칙(게이트 G1).** |rho| ≥ 0.5 → 전처리를 사인 피팅 잔차 기반으로 변경(G1-a). 아니면 세그먼트 평균 제거 유지(G1-b).

*주의.* f가 세그먼트 길이의 역수의 절반보다 작으면(반 주기도 안 들어감) 사인은 선형 추세와 구분되지 않아 A·C가 발산한다. 그런 피팅은 `trend_like=True`로 표시하고 상관 계산에서 제외한다(길이 10샘플 미만도 제외).

In [3]:
raw_u = pp.unify_format(raw)                         # 형식 통일만 (DC는 아직 제거 안 함)
fit = sc.sine_fit_table(raw_u, fs=10.0)               # 620 세그먼트 × 3채널, 약 20~30초
fit = fit.join(seg_meta, on="seg_uid")
fit["dc_abs"] = fit["dc_mean"].abs()
fit["dur_s"] = fit["n"] / 10.0
fit["trend_like"] = fit["f_hz"] < 0.5 / fit["dur_s"]   # 반 주기 미만 → 선형 추세와 구분 불가
fit.to_csv(RES / "sine_fit_by_segment.csv", index=False, encoding="utf-8-sig")
print("trend_like 비율:", fit.groupby(["channel", "src"])["trend_like"].mean().round(3).to_dict())
fit.head()

trend_like 비율: {('AI0_Vibration', 'normal'): 0.017, ('AI0_Vibration', 'outlier'): 0.048, ('AI1_Vibration', 'normal'): 0.007, ('AI1_Vibration', 'outlier'): 0.19, ('AI2_Current', 'normal'): 0.048, ('AI2_Current', 'outlier'): 0.333}


,seg_uid,channel,f_hz,amp,phase,offset,dc_mean,resid_rms,r2,n,src,seg_id,len,label,state,vib_grade,cur_grade,vib_rms,dc_abs,dur_s,trend_like
0,normal_0,AI0_Vibration,3.201612,0.059161,0.993356,-0.007206,-0.006957,0.054664,0.366107,41,normal,0,41,0.0,0,<NA>,<NA>,0.177171,0.006957,4.1,False
1,normal_0,AI1_Vibration,1.789226,0.220483,-2.226875,-0.000776,-0.009436,0.042100,0.933211,41,normal,0,41,0.0,0,<NA>,<NA>,0.177171,0.009436,4.1,False
2,normal_0,AI2_Current,0.600238,210.793615,1.143597,0.722328,19.044365,4.631965,0.999019,41,normal,0,41,0.0,0,<NA>,<NA>,0.177171,19.044365,4.1,False
3,normal_1,AI0_Vibration,3.601012,0.063316,0.833510,0.006001,0.007061,0.057821,0.373719,37,normal,1,37,0.0,0,<NA>,<NA>,0.151397,0.007061,3.7,False
4,normal_1,AI1_Vibration,1.793258,0.177099,0.856815,0.000660,0.006813,0.043472,0.891926,37,normal,1,37,0.0,0,<NA>,<NA>,0.151397,0.006813,3.7,False


In [4]:
# 파일 × 채널 요약: 피팅 f, 진폭, R², 잔차 RMS
ok = fit[(fit["n"] >= 10) & (~fit["trend_like"])]
summ = (ok.groupby(["channel", "src"])
          .agg(n_seg=("seg_uid", "size"), f_med=("f_hz", "median"), f_q25=("f_hz", lambda s: s.quantile(.25)),
               f_q75=("f_hz", lambda s: s.quantile(.75)), amp_med=("amp", "median"), r2_med=("r2", "median"),
               resid_rms_med=("resid_rms", "median"), dc_abs_med=("dc_abs", "median"))
          .round(4))
summ.to_csv(RES / "sine_fit_summary.csv", encoding="utf-8-sig")
summ

n_seg   f_med   f_q25   f_q75   amp_med  r2_med  resid_rms_med  dc_abs_med
channel       src                                                                                
AI0_Vibration normal     522  3.5533  1.7787  3.5983    0.0683  0.5144         0.0431      0.0010
              outlier     16  1.0546  0.8241  1.2021    0.3063  0.3509         0.2951      0.0420
AI1_Vibration normal     529  1.8000  1.7892  3.5716    0.1087  0.7698         0.0413      0.0010
              outlier     13  0.9886  0.6143  1.1345    0.1871  0.3503         0.1827      0.0360
AI2_Current   normal     528  0.6001  0.5987  0.6011  144.6357  0.9965         5.6640      2.8171
              outlier     11  0.4825  0.2081  2.5918   72.0600  0.4635        51.2037     74.3864

In [5]:
# 게이트 G1: 채널별 |DC| vs 피팅 f 의 Spearman 상관 (정상 / 이상 / 합침)
rows = []
for ch in pp.SENSORS:
    s = ok[ok["channel"] == ch]
    for name, g in [("normal", s[s["src"] == "normal"]), ("outlier", s[s["src"] == "outlier"]), ("pooled", s)]:
        rho, p = stats.spearmanr(g["dc_abs"], g["f_hz"])
        rows.append({"channel": ch, "subset": name, "n": len(g), "spearman_rho": round(rho, 3), "p_value": p,
                     "share_f_below_0.1Hz": round((g["f_hz"] < 0.1).mean(), 3)})
gate = pd.DataFrame(rows)
gate.to_csv(RES / "sine_fit_dc_vs_freq_spearman.csv", index=False, encoding="utf-8-sig")
cur_rho = gate.loc[(gate["channel"] == "AI2_Current") & (gate["subset"] == "pooled"), "spearman_rho"].iloc[0]
print("G1 판정:", "G1-a (에일리어싱 산물 → 사인 잔차 전처리)" if abs(cur_rho) >= 0.5 else f"G1-b (|rho|={abs(cur_rho):.3f} < 0.5 → 세그먼트 평균 제거 유지)")
gate

G1 판정: G1-b (|rho|=0.034 < 0.5 → 세그먼트 평균 제거 유지)


,channel,subset,n,spearman_rho,p_value,share_f_below_0.1Hz
0,AI0_Vibration,normal,522,-0.092,0.035005,0.0
1,AI0_Vibration,outlier,16,-0.218,0.418087,0.0
2,AI0_Vibration,pooled,538,-0.153,0.000385,0.0
3,AI1_Vibration,normal,529,-0.093,0.032636,0.0
4,AI1_Vibration,outlier,13,-0.022,0.943186,0.0
5,AI1_Vibration,pooled,542,-0.145,0.000704,0.0
6,AI2_Current,normal,528,-0.041,0.346298,0.0
7,AI2_Current,outlier,11,0.200,0.555445,0.0
8,AI2_Current,pooled,539,-0.034,0.424872,0.0


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, ch in zip(axes, pp.SENSORS):
    s = ok[ok["channel"] == ch]
    for src, c, m in [("normal", "tab:blue", "."), ("outlier", "tab:red", "x")]:
        g = s[s["src"] == src]
        ax.scatter(g["f_hz"], g["dc_abs"], s=14, c=c, marker=m, alpha=.6, label=f"{src} (n={len(g)})")
    r = gate[(gate["channel"] == ch) & (gate["subset"] == "pooled")].iloc[0]
    ax.set_xscale("log"); ax.set_xlabel("fitted f [Hz] (log)"); ax.set_ylabel("|DC| = |segment mean|")
    ax.set_title(f"{ch}: Spearman rho={r['spearman_rho']:.2f}"); ax.axvline(0.1, ls="--", c="gray", lw=.8); ax.legend(fontsize=8)
paths.save_fig(FIG, "sine_fit_freq_vs_dc.png")

saved figures\03_diagnosis_recheck_CHS\sine_fit_freq_vs_dc.png


### 이상 세그먼트 16~20의 정체: 진폭은 작고 오프셋만 크다

In [7]:
cur = fit[fit["channel"] == "AI2_Current"]
normal_amp = cur[(cur["src"] == "normal") & (cur["n"] >= 10) & (~cur["trend_like"])]["amp"]
o = cur[cur["src"] == "outlier"].sort_values("seg_id")[["seg_id", "n", "cur_grade", "f_hz", "amp", "offset", "dc_mean", "resid_rms", "r2", "trend_like"]]
o["amp_pct_of_normal"] = o["amp"].apply(lambda a: round(float((normal_amp < a).mean()), 3))   # 정상 진폭 분포에서의 백분위
o.round(3).to_csv(RES / "sine_fit_outlier_current.csv", index=False, encoding="utf-8-sig")
print("정상 전류 진폭 분위(10/50/90%):", normal_amp.quantile([.1, .5, .9]).round(1).tolist())
o.round(3)

정상 전류 진폭 분위(10/50/90%): [115.3, 144.6, 245.9]


,seg_id,n,cur_grade,f_hz,amp,offset,dc_mean,resid_rms,r2,trend_like,amp_pct_of_normal
1799,0,4,경계,4.999,19666.409,-17.136,-24.736,13.411,0.934,False,1.000
1802,1,50,경계,0.224,72.060,40.576,43.988,73.744,0.309,False,0.000
1805,2,47,경계,0.230,112.814,-16.936,-9.258,60.920,0.641,False,0.006
1808,3,31,경계,4.441,52.671,74.526,73.448,56.066,0.291,False,0.000
1811,4,18,경계,0.219,162.588,-19.500,-120.269,70.787,0.451,True,0.583
1814,5,3,경계,NaN,NaN,-141.859,-141.859,NaN,NaN,False,0.000
1817,6,50,경계,0.181,132.344,-83.592,-74.386,89.924,0.519,False,0.455
1820,7,40,경계,0.106,273.497,-253.212,-81.420,81.735,0.622,True,1.000
1823,8,25,경계,0.010,13943.987,-13904.349,-26.226,72.161,0.374,True,1.000
1826,9,8,경계,3.467,59.348,-114.089,-108.778,33.468,0.572,False,0.000


In [8]:
# 파형 + 피팅 예시: 정상 세그먼트(중앙값 f) vs 이상 17·19 (DC 큰 세그먼트)
def _fitted(row, n):
    t = np.arange(n) / 10.0
    return row["amp"] * np.sin(2 * np.pi * row["f_hz"] * t + row["phase"]) + row["offset"]
ex_normal = cur[(cur["src"] == "normal") & (cur["n"] == 50)].iloc[len(cur[(cur["src"] == "normal") & (cur["n"] == 50)]) // 2]
picks = [("normal", ex_normal["seg_uid"]), ("outlier", "outlier_17"), ("outlier", "outlier_19")]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (src, uid) in zip(axes, picks):
    x = raw_u.loc[raw_u["seg_uid"] == uid, "AI2_Current"].to_numpy()
    row = cur[cur["seg_uid"] == uid].iloc[0]
    ax.plot(np.arange(len(x)) / 10, x, "k.-", lw=.8, label="AI2_Current")
    ax.plot(np.arange(len(x)) / 10, _fitted(row, len(x)), "r-", lw=1.2, label=f"fit f={row['f_hz']:.3f}Hz A={row['amp']:.0f} C={row['offset']:.0f}")
    ax.axhline(row["dc_mean"], c="tab:blue", ls="--", lw=.8, label=f"mean={row['dc_mean']:.0f}")
    ax.set_title(f"{uid} (R²={row['r2']:.2f}, resid RMS={row['resid_rms']:.1f})", fontsize=9); ax.set_xlabel("s"); ax.legend(fontsize=7)
paths.save_fig(FIG, "sine_fit_examples.png")

saved figures\03_diagnosis_recheck_CHS\sine_fit_examples.png


### 부수 발견: 정상 전류는 0.600 Hz 순수 정현파, 이상은 정현파에서 벗어난다 → 사인 잔차가 피처 후보

In [9]:
# 전류 피팅 부산물의 단독 분리력 (세그먼트 단위 AUC, n>=10). 형식 누수와 무관한 "파형 왜곡" 지표.
c10 = cur[cur["n"] >= 10].copy()
c10["f_dev"] = (c10["f_hz"] - 0.6).abs()
c10["resid_ratio"] = c10["resid_rms"] / c10["amp"].clip(lower=1e-9)
auc_rows = []
for col in ["resid_rms", "r2", "f_dev", "resid_ratio", "dc_abs", "amp"]:
    a = roc_auc_score(c10["label"], c10[col]); auc_rows.append({"feature": col, "auc_sym": round(max(a, 1 - a), 4), "n_normal": int((c10.label == 0).sum()), "n_outlier": int((c10.label == 1).sum())})
auc_tab = pd.DataFrame(auc_rows).sort_values("auc_sym", ascending=False)
auc_tab.to_csv(RES / "sine_fit_feature_auc.csv", index=False, encoding="utf-8-sig")
auc_tab

,feature,auc_sym,n_normal,n_outlier
0,resid_rms,1.0000,530,17
1,r2,1.0000,530,17
2,f_dev,0.9992,530,17
4,dc_abs,0.9203,530,17
3,resid_ratio,0.8824,530,17
5,amp,0.6484,530,17


### A-1 결론 — 게이트 **G1-b**: DC 오프셋은 에일리어싱 산물이 아니다. 세그먼트 평균 제거를 유지한다

- 전류 |DC| vs 피팅 f Spearman rho: 정상 −0.041 (n=528, p=0.35) / 이상 +0.200 (n=11) / 합침 **−0.034** (n=539, p=0.42) → |rho| < 0.5. f가 낮을수록 |DC|가 커지는 관계가 없다. 진동도 −0.153(AI0)·−0.145(AI1)로 약해 진동 DC의 형식 누수 결론은 유지한다.
- 정상 전류는 세그먼트마다 **f = 0.6001 Hz(IQR 0.5987~0.6011), R² 0.9965, 진폭 중앙값 144.6**의 순수 정현파다. 02 §3의 zero-crossing 추정(0.57 Hz)보다 정밀하며, 60 Hz 구동이 10 Hz 샘플링에 접힌 겉보기 주파수가 0.600 Hz라는 뜻이다(실제 구동 주파수 후보 59.4 / 60.6 Hz — `alias_candidates` 참고). 정상 안에서 f 편차가 ±0.001 Hz에 불과하므로 "구동 주파수 이탈 → 느린 정현파 → DC"라는 경로는 정상 데이터에서 발생하지 않는다.
- 이상 세그먼트 17~20의 피팅 진폭은 138 / 99 / 57 / 29로 정상 분포의 47 / 0 / 0 / 0 백분위이고, 오프셋 C는 288 / 237 / 188 / 179다. 즉 **진폭은 줄고 오프셋만 크다** → "실제 오프셋"이다. 파형 그림(`sine_fit_examples.png`)에서 이상 17은 전류가 80~480으로 전부 양수이고 0.6 Hz 성분이 사라져 있다.
- 세그먼트 길이 대비 반 주기 미만인 피팅(`trend_like`, 이상 전류 33%, 정상 5%)은 A·C가 발산해 제외했다. 제외 후 남은 피팅에는 0.1 Hz 미만이 없다(길이 ≤5 s이면 판정 하한이 0.1 Hz이므로 정의상 그렇다).

**부수 발견(5번 창의성 후보).** 정상 전류가 정현파에 극히 잘 맞기 때문에 "정현파에서 얼마나 벗어났는가"가 그 자체로 이상 지표가 된다: 사인 피팅 잔차 RMS(정상 중앙값 5.66 vs 이상 51.2, 약 9배)와 R²(0.9965 vs 0.46)의 세그먼트 단위 AUC가 **1.000**(n=530 / 17), |f − 0.6| 0.999. 이는 1.19 격자 양자화(최대 오차 0.6)로는 만들 수 없는 크기라 형식 누수는 아니다. 다만 이상 이벤트가 1건(하루)뿐이라 "그날 전류 계측 경로가 달랐을 가능성"(날짜 교란)은 배제할 수 없으므로, 이 피처는 진폭 피처와 별도 모델로 두고 A-3 차단 실험과 3번 오류분석에서 재검토한다.

**모델 단계 반영.** `preprocess.remove_dc(method="mean")` 유지(`sine` 미구현). `features.py`에 전류 사인 피팅 부산물(`cur_fit_resid_rms`, `cur_fit_r2`, `cur_fit_f_dev`)을 **별도 피처군**으로 추가하되 기본 모델에는 넣지 않고 ablation으로 비교한다. 02 §3 "FFT·스펙트럼 분석 무의미" 표현은 "결함 주파수 분석 불가, 단 접힌 0.600 Hz 성분의 이탈은 지표 후보"로 정정한다(A-4).